In [1]:
import pandas as pd
import numpy as np

### Step 1. Load the Data

In [2]:
df_raw = pd.read_csv("tech_layoffs.csv")
df_raw.head()

,company,total_layoffs,impacted_workforce_percentage,reported_date,industry,headquarter_location,sources,status,additional_notes
0,C2FO,20,2,12/9/2022,"Fintech, payments","Leawood, KS",Kansas City Business Journal,Private,NaN
1,VideoAmp,Unclear,2,9/19/2022,Advertising platforms,Los Angeles,Company memo,Private,NaN
2,Amperity,13,3,8/20/2022,"Saas, analytics",Seattle,Geekwire,Private,NaN
3,Addepar,20,3,1/18/2023,"fintech, data analytics","Mountain View, CA",Fortune,Private,NaN
4,Thirty Madison,24,3,5/13/2022,Health Care,New York,Axios,Private,"Per Axios, 22 of the employees laid off were f..."


### Step 2. Audit (measuring mess before fixing it)

In [3]:
def audit(df, name='data'):
    print(f'{name}: {df.shape[0]} rows and {df.shape[1]} columns')
    report = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_%': (df.isna().mean() * 100).round(1),
        'unique': df.nunique()
    })
    print(report)
    print('Exact duplicate rows:', df.duplicated().sum())

audit(df_raw, 'RAW')

RAW: 489 rows and 9 columns
                              dtype  missing  missing_%  unique
company                         str        0        0.0     477
total_layoffs                   str        0        0.0     150
impacted_workforce_percentage   str        0        0.0      50
reported_date                   str        0        0.0     187
industry                        str        0        0.0     289
headquarter_location            str        0        0.0     134
sources                         str        0        0.0     221
status                          str        0        0.0       2
additional_notes                str      467       95.5      22
Exact duplicate rows: 0


### Step 3. Tidy column names

In [4]:
def step_columns(df):
    df = df.copy()
    df.columns = (df.columns.str.strip()
                            .str.lower()
                            .str.replace(r'\s+','_',regex=True))
    return df

df = step_columns(df_raw)
df.columns.tolist()


['company',
 'total_layoffs',
 'impacted_workforce_percentage',
 'reported_date',
 'industry',
 'headquarter_location',
 'sources',
 'status',
 'additional_notes']

### Step 4. Normalize text

In [5]:
def text_columns(df):
    return df.select_dtypes(include=['object','string']).columns.tolist()

def step_text(df):
    df = df.copy()
    for col in text_columns(df):
        df[col] = (df[col].str.strip()
                          .str.replace(r'\s+',' ',regex=True)
                          .str.lower())
    return df
df = step_text(df)
df[['company','industry','sources']].head(6)

,company,industry,sources
0,c2fo,"fintech, payments",kansas city business journal
1,videoamp,advertising platforms,company memo
2,amperity,"saas, analytics",geekwire
3,addepar,"fintech, data analytics",fortune
4,thirty madison,health care,axios
5,nextroll,marketing tech,business insider


### Step 5. Replacing sentinel values with NaN

In [6]:
def step_sentinals(df):
    df = df.copy()
    for col in text_columns(df):
        df[col] = df[col].replace('unclear',np.nan) # missing values are only represented by 'unclear'.
    return df
df = step_sentinals(df)
df

,company,total_layoffs,impacted_workforce_percentage,reported_date,industry,headquarter_location,sources,status,additional_notes
0,c2fo,20,2,12/9/2022,"fintech, payments","leawood, ks",kansas city business journal,private,NaN
1,videoamp,NaN,2,9/19/2022,advertising platforms,los angeles,company memo,private,NaN
2,amperity,13,3,8/20/2022,"saas, analytics",seattle,geekwire,private,NaN
3,addepar,20,3,1/18/2023,"fintech, data analytics","mountain view, ca",fortune,private,NaN
4,thirty madison,24,3,5/13/2022,health care,new york,axios,private,"per axios, 22 of the employees laid off were f..."
...,...,...,...,...,...,...,...,...,...
484,salesforce,9090,NaN,1/4/2023,"enterprise tech, analytics",san francisco,new york times,public,NaN
485,oracle,NaN,NaN,8/1/2022,"saas, enterprise software",austin,the information,public,the information reported thousands of people w...
486,liveperson,NaN,NaN,9/15/2022,customer service,new york,linkedin,public,NaN
487,skillz,NaN,NaN,8/29/2022,"mobile apps, gaming",san francisco,seeking alpha,public,NaN


### Step 6. Removing duplicates

In [7]:
before_norm = step_columns(df_raw).duplicated().sum()
after_norm = df.duplicated().sum()
print('Exact duplicates before normalizing text:',before_norm)
print('Exact duplicates after normalizing text:',after_norm)

Exact duplicates before normalizing text: 0
Exact duplicates after normalizing text: 0


In [8]:
df[df.duplicated(subset=['company', 'reported_date'], keep=False)][['company', 'reported_date', 'total_layoffs', 'industry']]

,company,reported_date,total_layoffs,industry
135,socure,1/5/2023,104,"cybsersecurity, predictive analytics"
201,socure,1/5/2023,173,identity verification


In [9]:
def step_dedupe(df):
    df = df.copy()
    df = df.drop_duplicates()   # exact duplicates
    df = df.drop_duplicates(
        subset=['company', 'reported_date', 'total_layoffs'],
        keep='last'
    )   
    return df.reset_index(drop=True)

n_before = len(df)
df = step_dedupe(df)
print(f'Rows: {n_before} -> {len(df)}')

df[df['company'] == 'socure'][['company', 'reported_date', 'total_layoffs']]

Rows: 489 -> 489


,company,reported_date,total_layoffs
135,socure,1/5/2023,104
201,socure,1/5/2023,173


### Step 7. Turning text into numbers

In [10]:
def step_numbers(df):
    df = df.copy()
    df['total_layoffs'] = pd.to_numeric(df['total_layoffs'], errors='coerce')
    df['impacted_workforce_percentage'] = pd.to_numeric(
        df['impacted_workforce_percentage'], errors='coerce')
    return df

df = step_numbers(df)
df[['total_layoffs', 'impacted_workforce_percentage']].head(8)


,total_layoffs,impacted_workforce_percentage
0,20.0,2.0
1,NaN,2.0
2,13.0,3.0
3,20.0,3.0
4,24.0,3.0
5,NaN,3.0
6,25.0,4.0
7,26.0,4.0


In [11]:
df[['total_layoffs','impacted_workforce_percentage']].isna().sum()

total_layoffs                    182
impacted_workforce_percentage    133
dtype: int64

### Step 8. Splitting one column into fields

In [12]:
LOCATION = r'^(?P<city>[^,]+?)(?:,\s*(?P<region>.+))?$'

def step_location(df):
    df = df.copy()
    parts = df['headquarter_location'].str.extract(LOCATION)
    df['hq_city'] = parts['city'].str.title()
    df['hq_region'] = parts['region'].str.upper()
    return df.drop(columns=['headquarter_location'])

before_loc = df['headquarter_location'].copy()
df = step_location(df)

print('Locations with no region (no comma in the text):')
before_loc[df['hq_region'].isna()].head(10)

Locations with no region (no comma in the text):


1       los angeles
2           seattle
4          new york
5     san francisco
6     san francisco
7           chicago
8       los angeles
9           seattle
11    mountain view
12    san francisco
Name: headquarter_location, dtype: str

### Step 9. Standardize Categories

In [13]:
city_fix = {
    'San Franicsco': 'San Francisco',
    'San Francsico': 'San Francisco',
    'Dever': 'Denver',
    'New York City': 'New York',
    'Greater New York Area': 'New York',
    'Playa Visa': 'Playa Vista',
    'Distributed': np.nan,        # remote company, its not a real city
}

region_fix = {
    'CALIFORNIA': 'CA', 'NEW JERSEY': 'NJ', 'OREGON': 'OR',
    'UTAH': 'UT', 'MASSACHUSETTS': 'MA', 'NORTH CAROLINA': 'NC',
    'VIRGINIA': 'VA',
}

def step_standardize(df):
    df = df.copy()
    df['hq_city'] = df['hq_city'].replace(city_fix)
    df['hq_region'] = df['hq_region'].replace(region_fix)
    return df

df = step_standardize(df)

print(df['hq_city'].value_counts().head(15))
print(df['hq_region'].value_counts().head(15))

hq_city
San Francisco    139
New York          81
Seattle           21
Los Angeles       20
Boston            16
Chicago           12
Austin            10
Mountain View      9
Palo Alto          9
San Jose           8
San Mateo          6
San Diego          5
Atlanta            5
Miami              4
Boulder            4
Name: count, dtype: int64
hq_region
CA    55
MA     8
TX     6
UT     5
WA     4
NY     4
DC     3
MI     3
FL     3
NC     3
OR     2
CT     2
NJ     2
NE     2
DE     2
Name: count, dtype: int64


In [14]:
industry_fix = {
    'cybsersecurity': 'cybersecurity',
    'transporation': 'transportation',
    'ecommerce': 'e-commerce',
    'healthcare': 'health care',
}

group = {
    'crypto': 'cryptocurrency',
    'blockchain': 'cryptocurrency',
    'ai': 'artificial intelligence',
    'saas': 'enterprise software',
}

def step_industry(df):
    df = df.copy()
    for wrong, right in industry_fix.items():
        df['industry'] = df['industry'].str.replace(wrong, right, regex=False)
    df['main_industry'] = df['industry'].str.split(',').str[0].str.strip()
    df['main_industry'] = df['main_industry'].replace(group)
    return df

df = step_industry(df)
df['main_industry'].value_counts().head(15)

main_industry
fintech                    48
health care                33
proptech                   30
e-commerce                 29
cryptocurrency             23
enterprise software        21
artificial intelligence    18
cybersecurity              14
edtech                     14
insurtech                  12
transportation             10
social media               10
media                       7
media/entertainment         7
food delivery               6
Name: count, dtype: int64

### Step 10. Validate formats

In [15]:
pattern = r'^\d{1,2}/\d{1,2}/\d{4}$'
valid = df['reported_date'].astype(str).str.match(pattern)
print('Dates not in M/D/YYYY format:', (~valid).sum())
df[~valid]['reported_date'].head()

Dates not in M/D/YYYY format: 0


Series([], Name: reported_date, dtype: str)

### Step 11. Domain Rules

In [16]:
RULES = {
    'total_layoffs':                 (0, 50_000),
    'impacted_workforce_percentage': (0, 100),
}

def step_validity(df):
    df = df.copy()
    for col, (low, high) in RULES.items():
        bad = df[col].notna() & ~df[col].between(low, high)
        print(f'{col:30s} -> {int(bad.sum())} value(s) outside [{low:,}, {high:,}]')
        df.loc[bad, col] = np.nan
    return df

df = step_validity(df)

total_layoffs                  -> 0 value(s) outside [0, 50,000]
impacted_workforce_percentage  -> 0 value(s) outside [0, 100]


### Step 12. Handling missing values

In [17]:
df.isna().sum()

company                            0
total_layoffs                    182
impacted_workforce_percentage    133
reported_date                      0
industry                           0
sources                            0
status                             0
additional_notes                 467
hq_city                            1
hq_region                        367
main_industry                      0
dtype: int64

| Column | Missing | Decision | Reason |
|---|---|---|---|
| `total_layoffs` | 182 | Leave as `NaN` | Companies did not report it and filling would invent layoffs |
| `impacted_workforce_percentage` | 133 | Leave as `NaN` | Same reason |
| `additional_notes` | 467 | Leave as `NaN` | it is not used in analysis |
| `hq_region` | 367 | Leave as `NaN` | Many cities simply have no region |
| `hq_city` | 1 | Fill with `Unknown` | It is the `Distributed` company and filling keeps the row |

In [18]:
def step_missing(df):
    df = df.copy()
    df['hq_city'] = df['hq_city'].fillna('Unknown')
    return df

df = step_missing(df)
df[['hq_city', 'total_layoffs', 'impacted_workforce_percentage']].isna().sum()

hq_city                            0
total_layoffs                    182
impacted_workforce_percentage    133
dtype: int64

### Step 13. Outliers

In [19]:
def step_outliers(df):
    df = df.copy()
    q1, q3 = df['total_layoffs'].quantile(0.25), df['total_layoffs'].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df['layoffs_outlier'] = (df['total_layoffs'] < low) | (df['total_layoffs'] > high)
    print(f'IQR limits: {low:,.0f} to {high:,.0f}')
    print('Outliers flagged:', df['layoffs_outlier'].sum())
    return df

df = step_outliers(df)
df.loc[df['layoffs_outlier'], ['company', 'industry', 'reported_date', 'total_layoffs']] \
  .sort_values('total_layoffs', ascending=False).head(15)

IQR limits: -190 to 434
Outliers flagged: 46


,company,industry,reported_date,total_layoffs
378,amazon,"e-commerce, saas",1/5/2023,18000.0
470,alphabet,"big tech, internet services",1/20/2023,12000.0
420,meta,social media,11/9/2022,11000.0
469,microsoft,enterprise software,1/18/2023,10000.0
484,salesforce,"enterprise tech, analytics",1/4/2023,9090.0
224,better.com,fintech,4/19/2022,5000.0
381,cisco,"enterprise software, saas",11/16/2022,4100.0
475,peloton,fitness,10/6/2022,4084.0
466,carvana,"automotive, e-commerce",1/13/2023,4000.0
463,twitter,social media,1/5/2023,3740.0


### Step 14. Fixing data types

In [20]:
def step_types(df):
    df = df.copy()
    df['total_layoffs'] = df['total_layoffs'].round().astype('Int64')
    df['impacted_workforce_percentage'] = df['impacted_workforce_percentage'].round().astype('Int64')
    df['reported_date'] = pd.to_datetime(df['reported_date'], format='%m/%d/%Y', errors='coerce')
    for col in ['status', 'main_industry', 'hq_city', 'hq_region']:
        df[col] = df[col].astype('category')
    return df

df = step_types(df)
df.dtypes
print('Bad dates:', df['reported_date'].isna().sum())
print('Date range:', df['reported_date'].min(), 'to', df['reported_date'].max())
df.dtypes

Bad dates: 0
Date range: 2022-01-27 00:00:00 to 2023-01-24 00:00:00


company                                     str
total_layoffs                             Int64
impacted_workforce_percentage             Int64
reported_date                    datetime64[us]
industry                                    str
sources                                     str
status                                 category
additional_notes                            str
hq_city                                category
hq_region                              category
main_industry                          category
layoffs_outlier                            bool
dtype: object

### Step 15. Feature Engineering

In [21]:
def step_features(df):
    df = df.copy()
    df['month'] = df['reported_date'].dt.to_period('M').astype(str)
    df['quarter'] = df['reported_date'].dt.to_period('Q').astype(str)
    df['layoff_size'] = pd.cut(df['total_layoffs'],
                               [0, 50, 200, 1000, np.inf],
                               labels=['Small', 'Medium', 'Large', 'Huge'],
                               include_lowest=True)   # chosen EDGES
    df['impact_level'] = pd.cut(df['impacted_workforce_percentage'],
                                [0, 5, 15, 100],
                                labels=['Low', 'Medium', 'High'],
                                include_lowest=True)  # chosen EDGES
    return df

df = step_features(df)
df[['company', 'reported_date', 'month', 'total_layoffs', 'layoff_size',
    'impacted_workforce_percentage', 'impact_level']].head(8)

,company,reported_date,month,total_layoffs,layoff_size,impacted_workforce_percentage,impact_level
0,c2fo,2022-12-09,2022-12,20,Small,2,Low
1,videoamp,2022-09-19,2022-09,<NA>,NaN,2,Low
2,amperity,2022-08-20,2022-08,13,Small,3,Low
3,addepar,2023-01-18,2023-01,20,Small,3,Low
4,thirty madison,2022-05-13,2022-05,24,Small,3,Low
5,nextroll,2022-07-08,2022-07,<NA>,NaN,3,Low
6,armis security,2022-12-08,2022-12,25,Small,4,Low
7,foxtrot,2022-09-23,2022-09,26,Small,4,Low


### Step 16. Encoding

In [22]:
model_df = pd.concat([
    df[['total_layoffs', 'impacted_workforce_percentage']].astype('float64'),
    pd.get_dummies(df[['status', 'main_industry']], drop_first=True, dtype=int),
], axis='columns')
model_df.head()

,total_layoffs,impacted_workforce_percentage,status_public,main_industry_adtech,main_industry_advertising platforms,main_industry_aerospace,main_industry_agtech,main_industry_analytics,main_industry_ar,main_industry_artificial intelligence,...,main_industry_travel,main_industry_vertical farming,main_industry_video conferencing software,main_industry_video games,main_industry_video production,main_industry_virtual event,main_industry_virtual events,main_industry_visual collaboration,main_industry_wellness,main_industry_wifi
0,20.0,2.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,NaN,2.0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13.0,3.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,20.0,3.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,24.0,3.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Step 17. Splitting


In [23]:
model_df = model_df.dropna()

In [24]:
train = model_df.sample(frac=0.8, random_state=42)
test = model_df.drop(train.index)
print('train:', train.shape, '| test:', test.shape)

train: (192, 146) | test: (48, 146)


### Step 18. Validate

In [25]:
audit(df_raw, 'BEFORE')   # the messy original
print()                
audit(df, 'AFTER')     # the cleaned data

BEFORE: 489 rows and 9 columns
                              dtype  missing  missing_%  unique
company                         str        0        0.0     477
total_layoffs                   str        0        0.0     150
impacted_workforce_percentage   str        0        0.0      50
reported_date                   str        0        0.0     187
industry                        str        0        0.0     289
headquarter_location            str        0        0.0     134
sources                         str        0        0.0     221
status                          str        0        0.0       2
additional_notes                str      467       95.5      22
Exact duplicate rows: 0

AFTER: 489 rows and 16 columns
                                        dtype  missing  missing_%  unique
company                                   str        0        0.0     477
total_layoffs                           Int64      182       37.2     149
impacted_workforce_percentage           Int64      

In [26]:
essential = ['company', 'reported_date', 'main_industry', 'status']

assert not df.duplicated(subset=['company', 'reported_date', 'total_layoffs']).any(), \
    'duplicate layoff record remains'
assert df[essential].notna().all().all(), \
    'missing values in essential columns'
assert df['total_layoffs'].dropna().between(*RULES['total_layoffs']).all(), \
    'total_layoffs outside valid range'
assert df['impacted_workforce_percentage'].dropna().between(*RULES['impacted_workforce_percentage']).all(), \
    'percentage outside valid range'
assert df['reported_date'].between('2022-01-01', '2023-12-31').all(), \
    'date outside expected range'
assert not df['hq_city'].isin(['San Franicsco', 'San Francsico', 'Dever']).any(), \
    'city typo remains'
print('All checks passed.')

All checks passed.


In [27]:
df.to_csv('layoffs_clean.csv', index=False)